# E1.7 · Continuous control verification

**Function E — Governance, Risk, Compliance & the CISO Office → The GRC Practitioner (Risk & Control)**  ·  *Security of AI*

---

**Risk.** Automating judgment instead of evidence collection.

**Control.** Agent-assisted evidence collection, drift detection, exception tracking.

**This lab.** Automate the evidence package, not the judgment.

| | |
|---|---|
| Open-source tooling | OPA, OSCAL |
| Open-weight models | GLM-4.6 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("E1.7"))

Continuous control verification. The number that matters is not how much passed once — it is how much is *currently evidenced*.

In [ ]:
from cybercommons import grc
import time

now = time.time()
required = [c.cid for c in grc.CATALOGUE]
tests = [
    grc.ControlTest("AC-1", True,  "act chain in gateway logs",  tested_at=now - 2 * 86400),
    grc.ControlTest("AC-2", True,  "delegation refusal test",    tested_at=now - 9 * 86400),
    grc.ControlTest("SB-1", True,  "egress denial evidence",     tested_at=now - 31 * 86400),
    grc.ControlTest("SB-2", True,  "approval gate screenshot",   tested_at=now - 120 * 86400),
    grc.ControlTest("EV-1", True,  "audit sample, 50 actions",   tested_at=now - 5 * 86400),
    grc.ControlTest("EV-2", True,  "expert accuracy 0.91",       tested_at=now - 12 * 86400),
    grc.ControlTest("DR-1", False, "drift alerting not deployed", tested_at=now),
]
v = grc.verify_continuously(tests, required, now=now)
for r in v["rows"]:
    print(f"{r['control']:8s}{r['state']:14s}{r['age_days']}")
print(f"\ncoverage {v['coverage']:.1%} — {v['currently_evidenced']}/{v['required']}")

Two controls quietly aged out of their window. One was never deployed. One has no evidence at all. A point-in-time report shows 6/8 passing; the continuous view shows 4/8, and the difference is entirely made of things nobody did wrong — time simply passed.

### Expect

AC-1, AC-2, EV-1 and EV-2 are PASS; SB-1 and SB-2 are STALE; DR-1 is FAIL; ST-1 has NO EVIDENCE — coverage 50%.

### Your turn

Automate one of these tests so it re-runs weekly and writes its own `ControlTest`. That single change converts an annual assertion into a live control.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/E1.7.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*